In [24]:
import os
import re
import csv
import pandas as pd
from numpy import trapezoid


def parse_graph_stats(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    name_match = re.search(r'графе (.+?):', content)
    graph_name = name_match.group(1).strip() if name_match else os.path.basename(filepath)

    vertices = int(re.search(r'Количество вершин: (\d+)', content).group(1))
    edges = int(re.search(r'Количество рёбер: (\d+)', content).group(1))
    density = float(re.search(r'Плотность графа: ([\d.]+)', content).group(1))
    count_of_wcc = int(re.search(r'Количество WCC: (\d+)', content).group(1))
    v_largest_wcc = float(re.search(r'Доля вершин в наибольшей WCC: ([\d.]+)', content).group(1))
    count_of_scc = int(re.search(r'Количество SCC: (\d+)', content).group(1))
    v_largest_scc = float(re.search(r'Доля вершин в наибольшей SCC: ([\d.]+)', content).group(1))
    diameter_ds = int(re.search(r'Диаметр наибольшей WCC \(The Double Sweep\): (\d+)', content).group(1))
    percentile_ds = float(re.search(r'90 процентиль расстояний: ([\d.]+)', content).group(1))
    diameter_s = int(re.search(r'Диаметр наибольшей WCC \(Snowball \+ Double Sweep\): (\d+)', content).group(1))
    percentile_s = float(re.search(r'90 процентиль расстояний \(Snowball\): ([\d.]+)', content).group(1))
    triangles = int(re.search(r'Количество треугольников: (\d+)', content).group(1))
    avg_clustering = float(re.search(r'Средний коэффициент кластеризации: ([\d.]+)', content).group(1))
    global_clustering = float(re.search(r'Глобальный коэффициент кластеризации: ([\d.]+)', content).group(1))
    avg_clustering_wcc = float(re.search(r'Средний коэффициент кластеризации \(largest WCC\): ([\d.]+)', content).group(1))
    min_deg = float(re.search(r'Минимальная степень узлов: (\d+)', content).group(1))
    avg_deg = float(re.search(r'Средняя степень узлов: ([\d.]+)', content).group(1))
    max_deg = float(re.search(r'Максимальная степень узлов: (\d+)', content).group(1))

    def extract_block(header):
        match = re.search(header + r':\s*((?:\n\t+x = [^\n]+)+)', content)
        return match.group(1) if match else ''

    def extract_removal_data(block):
        data = {}
        for match in re.findall(r'x = ([\d.]+)%: доля вершин в наибольшей WCC: ([\d.]+)', block):
            x = float(match[0])
            y = float(match[1])
            data[x] = y
        return data

    block_random = extract_block(r'Удаление случайных узлов')
    block_degree = extract_block(r'Удаление узлов наибольшей степени')

    data_random = extract_removal_data(block_random)
    data_degree = extract_removal_data(block_degree)

    def compute_auc(data):
        if not data: return None
        x = sorted(data.keys())
        y = [data[k] for k in x]
        return trapezoid(y, x)

    auc_random = compute_auc(data_random)
    auc_degree = compute_auc(data_degree)

    return {
        'graph': graph_name,
        'vertices': vertices,
        'edges': edges,
        'density': density,
        'count_of_wcc': count_of_wcc,
        'v_largest_wcc': v_largest_wcc,
        'count_of_scc': count_of_scc,
        'v_largest_scc': v_largest_scc,
        'diameter_ds': diameter_ds,
        'percentile_ds': percentile_ds,
        'diameter_s': diameter_s,
        'percentile_s': percentile_s,
        'triangles': triangles,
        'avg_clustering': avg_clustering,
        'global_clustering': global_clustering,
        'avg_clustering_wcc': avg_clustering_wcc,
        'min_deg': min_deg,
        'avg_deg': avg_deg,
        'max_deg': max_deg,
        'auc_random': auc_random,
        'auc_degree': auc_degree
    }



input_dir = r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\output'
output_csv = os.path.join(r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization', 'graph_resilience.csv')

results = []

for file in os.listdir(input_dir):
    if file.endswith('.txt'):
        full_path = os.path.join(input_dir, file)
        stats = parse_graph_stats(full_path)
        results.append(stats)


with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        'graph', 'vertices', 'edges', 'density', 'count_of_wcc', 'v_largest_wcc', 
        'count_of_scc', 'v_largest_scc', 'diameter_ds', 'percentile_ds', 'diameter_s',
        'percentile_s', 'triangles', 'avg_clustering', 'global_clustering', 'avg_clustering_wcc',
        'min_deg', 'avg_deg', 'max_deg', 'auc_random', 'auc_degree'
    ])
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("CSV сохранён:", output_csv)

df = pd.read_csv(output_csv)
display(df)


CSV сохранён: C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization\graph_resilience.csv


,graph,vertices,edges,density,count_of_wcc,v_largest_wcc,count_of_scc,v_largest_scc,diameter_ds,percentile_ds,...,percentile_s,triangles,avg_clustering,global_clustering,avg_clustering_wcc,min_deg,avg_deg,max_deg,auc_random,auc_degree
0,CA-AstroPh,18772,396160,0.001124,290,0.953708,290,0.953708,14,6.0,...,5.0,1351441,0.6306,0.3179,0.6328,0.0,21.10,504.0,64.16075,21.497750
1,ca-coauthors-dblp,540486,15245729,0.000104,1,1.000000,1,1.000000,23,7.0,...,5.0,444095058,0.8019,0.6562,0.8019,1.0,56.41,3299.0,71.48700,46.063000
2,CA-GrQc,5242,28980,0.001055,355,0.793209,355,0.793209,17,8.0,...,7.0,48260,0.5296,0.6297,0.5569,0.0,5.53,81.0,35.76425,2.038000
3,com-orkut.ungraph,3072441,117185083,0.000025,1,1.000000,1,1.000000,9,5.0,...,3.0,627584181,0.1666,0.0413,0.1666,1.0,76.28,33313.0,77.68450,54.879250
4,com-youtube.ungraph,1134890,2987624,0.000005,1,1.000000,1,1.000000,24,7.0,...,4.0,3056386,0.0808,0.0062,0.0808,1.0,5.27,28754.0,45.92450,3.143976
5,Email-EuAll,265214,420045,0.000010,15836,0.847738,231000,0.128964,14,5.0,...,2.0,267313,0.0671,0.0041,0.0791,0.0,2.75,7636.0,7.14980,0.090664
6,musae_git_edges,37700,289003,0.000407,1,1.000000,37700,0.000027,11,4.0,...,2.0,523810,0.1675,0.0124,0.1675,1.0,15.33,9458.0,66.19150,9.118000
7,soc-wiki-Vote,889,2914,0.007383,1,1.000000,889,0.001125,13,6.0,...,5.0,2119,0.1528,0.1273,0.1528,1.0,6.56,102.0,55.87100,11.151250
8,vk,3215720,17414510,0.000003,24337,0.983362,3215720,0.000000,19,7.0,...,6.0,108030337,0.0494,0.1095,0.0499,1.0,10.83,6503.0,60.08175,12.474500
9,web-Google,875713,5105039,0.000011,2746,0.977263,371764,0.496530,24,9.0,...,5.0,13391903,0.5143,0.0552,0.5190,1.0,9.87,6332.0,53.05725,5.091250


**На основе таблицы можно сделать следующий вывод:**
1. Минимальная степень узлов равна 0, то есть графы могут содержать изолированные вершины.
2. Столбцы *auc_random* и *auc_degree* показывают площадь под кривой при удалении случайных узлов и удалении узлов наибольшей степени. Чем выше это значение, тем устойчивее граф к данному типу удаления.
3. Наиболее устойчивыми графами являются *com-orkut.ungraph* и *ca-coauthors-dbip*, а наименее - Email-EuAll.
4. Можно заметить, что наиболее устойчивые графы имеют наивысшую среднюю степень узлов. С наименее устойчивыми ситуация та же.